In [ ]:
# Import Libraries
!pip install faker
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta

In [ ]:
# initialize the environment
fake = Faker()

np.random.seed(42)
random.seed(42)

N_USERS = 100000

**Create the USERS Table**

Users include acquisition channels and device context.

In [ ]:
sources = [
    "organic",
    "google_ads",
    "facebook_ads",
    "tiktok",
    "referral",
    "affiliate"
]

countries = [
    "Kenya",
    "Nigeria",
    "South Africa",
    "UK",
    "India"
]

users = pd.DataFrame({

    "user_id":[f"U{i:07d}" for i in range(N_USERS)],

    "signup_time":[fake.date_time_between(
        start_date="-90d",
        end_date="now"
    ) for _ in range(N_USERS)],

    "age":np.random.randint(18,65,N_USERS),

    "country":np.random.choice(countries,N_USERS),

    "acquisition_source":np.random.choice(
        sources,
        N_USERS,
        p=[0.35,0.2,0.15,0.1,0.15,0.05]
    ),

    "fraud_risk_score":np.random.beta(2,8,N_USERS)
})

**Device Table**

Devices strongly influence KYC success.

Low camera score means selfie failure probability increases.

In [ ]:
devices = pd.DataFrame({

    "user_id":users["user_id"],

    "device_type":np.random.choice(
        ["mobile","web"],
        N_USERS,
        p=[0.85,0.15]
    ),

    "os":np.random.choice(
        ["Android","iOS"],
        N_USERS,
        p=[0.7,0.3]
    ),

    "device_model":np.random.choice([
        "Samsung A12",
        "iPhone 11",
        "iPhone 13",
        "Tecno Spark",
        "Huawei P30",
        "Infinix Note"
    ],N_USERS),

    "camera_quality_score":np.random.randint(3,10,N_USERS)
})

**Network Table**

Network conditions impact upload success.

In [ ]:
network_logs = pd.DataFrame({

    "user_id":users["user_id"],

    "network_type":np.random.choice(
        ["wifi","4G","5G","3G"],
        N_USERS,
        p=[0.4,0.4,0.1,0.1]
    ),

    "latency_ms":np.random.normal(120,40,N_USERS).clip(20,400),

    "packet_loss":np.random.beta(1,20,N_USERS),

    "upload_speed_mbps":np.random.normal(12,5,N_USERS).clip(1,50)
})

**Session Table**

Users may retry KYC multiple times. Rows ≈ 150k sessions

In [ ]:
sessions = []

session_id = 1

for uid in users["user_id"]:

    n_sessions = np.random.randint(1,3)

    for i in range(n_sessions):

        sessions.append({
            "session_id":f"S{session_id}",
            "user_id":uid,
            "session_start":fake.date_time_between("-60d","now"),
            "time_of_day":np.random.choice([
                "morning",
                "afternoon",
                "evening",
                "night"
            ])
        })

        session_id += 1

sessions = pd.DataFrame(sessions)

**KYC Event Generator**

This is the core dataset. Rows ≈ 250k KYC events

In [ ]:
kyc_steps = [
"start_kyc",
"phone_verification",
"personal_information",
"document_upload",
"document_validation",
"selfie_capture",
"face_match",
"address_verification",
"manual_review",
"kyc_approved"
]

events = []
event_id = 1

for _,session in sessions.iterrows():

    step_count = random.randint(3,len(kyc_steps))

    steps = kyc_steps[:step_count]

    for step in steps:

        status = np.random.choice(
            ["success","fail","abandon"],
            p=[0.75,0.15,0.10]
        )

        events.append({
            "event_id":event_id,
            "user_id":session["user_id"],
            "session_id":session["session_id"],
            "kyc_step":step,
            "status":status,
            "duration_seconds":np.random.randint(5,120),
            "retry_count":np.random.randint(0,3),
            "error_code":random.choice([
                None,
                "blurry_document",
                "face_mismatch",
                "network_timeout",
                "document_rejected"
            ])
        })

        event_id += 1

        if status == "abandon":
            break

kyc_events = pd.DataFrame(events)

**Add Realistic Failure Logic (Critical)**

Make failures depend on device/network.

Example:

Selfie failure probability increases if:

```
camera_quality < 5
latency > 200ms
network = 3G
```



In [ ]:
if step == "selfie_capture":

    if camera_quality < 5:
        fail_prob = 0.35
    else:
        fail_prob = 0.12

**Save Dataset**

In [ ]:
users.to_csv("fintech_users.csv",index=False)
devices.to_csv("fintech_devices.csv",index=False)
network_logs.to_csv("network_logs.csv",index=False)
sessions.to_csv("sessions.csv",index=False)
kyc_events.to_csv("kyc_events.csv",index=False)